# Native PyTorch tutorial 7

Original 5,000-epoch baseline and 50,000-epoch pruning workload. Prune only the final ParameterNet kernel, increasing sparsity from 10% to 90% with the cubic schedule. Native checkpoints and TorchScript replace legacy HDF5/TFLite. CPU packed int8 Linear inference is run and validated after export; generated ShapeNet computation remains floating point.

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial7"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() if isinstance(v, torch.Tensor) else v for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None, after_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        if after_epoch is not None:
            after_epoch(epoch)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# Model pruning

In this tutorial, you will learn how to prune/quantanzation your NIF model using [nif.pruning.PolynomialMagnitudePruning](../docs/PYTORCH_API.md). 

## First, let's train a model to start with

In [ ]:
cfg_shape_net = {"use_resblock": False, "connectivity": "full", "input_dim": 1,
                 "output_dim": 1, "units": 30, "nlayers": 2,
                 "weight_init_factor": 0.01, "omega_0": 30.0}
cfg_parameter_net = {"use_resblock": False, "input_dim": 1, "latent_dim": 1,
                     "units": 30, "nlayers": 2, "activation": "swish"}
def scheduler(epoch, lr):
    if epoch < 1000: return lr
    if epoch < 2000: return 1e-4
    if epoch < 4000: return 5e-5
    return 1e-5
nepoch,lr,batch_size = 5000,1e-3,512


In [ ]:
from nif.demo import TravelingWaveHighFreq
tw = TravelingWaveHighFreq()
train_data = tw.data
train_dataset = loader(train_data[:, :2], train_data[:, 2:3], batch_size=512)
# Shuffling is handled by DataLoader, keeping the source grid available for diagnostics.
model_ori=nif.NIFMultiScale(cfg_shape_net,cfg_parameter_net,"float32").to(device)
model=model_ori.build()
optimizer=nif.optimizers.ReferenceAdam(model.parameters(),lr=lr,eps=1e-7)
history_baseline=train(model,train_dataset,optimizer,epochs(nepoch),"baseline",schedule=scheduler)


### we save the prediction of trained model

In [ ]:
t=np.linspace(-1,1,10,endpoint=True)
x=np.linspace(-1,1,200,endpoint=True)
tt,xx=np.meshgrid(t,x)
input_feature=np.column_stack([tt.ravel(),xx.ravel()])
model=save_and_check(model,optimizer,"original",input_feature[:64])
y_pred_unpruned=predict(model,input_feature).reshape(200,10)


### We save this model as native state_dict format

In [ ]:
original_file=OUT / "original.pt"


## Second, let's load this model from disk and start pruning

- here we prune the ParameterNet only.
- only weights are pruned.
- you can choose to prune either the last layer of ParameterNet OR the whole ParameterNet
- when pruning, we use mag. based pruning criterion - low mag. weights are pruned

About pruning setup
- we use the native cubic magnitude-pruning schedule to prune 10% weights, and gradually increasing to 90%.
- the cubic schedule starts at 10%, but its first mask refresh is before training step 4,000 (epoch 1,000), after 3,999 dense steps. It refreshes every 4,000 steps, for 50 refreshes, reaching 90% before step 200,000. We remask at every epoch end without an extra final refresh.

In [ ]:
model_for_pruning=copy.deepcopy(model)
pruning_epochs=50000
prune_epoch=1000
# PolynomialDecay's cubic schedule: begin at 10%, end at 90% of the selected kernel.
# Full mode refreshes before steps 4,000 through 200,000 (50 updates).
# Steps 1 through 3,999 stay dense; no extra refresh follows training.


### Let's take a look each layer's name

In [ ]:
print(model_for_pruning)


- Here we can choose to prune what we want... how about start with last layer of ParameterNet? 
- We can write a function to pull out this layer by just checking which layer has **SIREN** in its name

In [ ]:
from nif.pruning import PolynomialMagnitudePruning
from nif.layers import HyperLinearForSIREN
selected=[(name,layer) for name,layer in model_for_pruning.named_modules()
          if isinstance(layer,HyperLinearForSIREN)]
assert len(selected)==1
selected_name,hyper_siren_layer=selected[0]
print("layer selected =",selected_name,"kernel",tuple(hyper_siren_layer.w.shape))

run_pruning_epochs=epochs(pruning_epochs)
steps_per_epoch=len(train_dataset)
# The separate smoke schedule covers the same stages in two development epochs.
# Full defaults: 2,000 rows / batch 512 -> 4 steps; end=200,000, frequency=4,000.
pruning_controller=PolynomialMagnitudePruning(hyper_siren_layer,"w",
    initial_sparsity=.10,final_sparsity=.90,begin_step=0,
    end_step=run_pruning_epochs*steps_per_epoch,power=3,
    frequency=(1 if SMOKE else prune_epoch)*steps_per_epoch)
# Masks are assigned before each forward, with raw gradients and optimizer slots
# active at masked coordinates. Refresh ranks actual weights, allowing reselection.


- get a redefined model for pruning

In [ ]:
optimizer_prune=nif.optimizers.ReferenceAdam(model_for_pruning.parameters(),lr=1e-4,eps=1e-7)


- Ok, now let's just prune it - first prune, then retrain, then prune....

In [ ]:
history_prune=train(model_for_pruning,train_dataset,optimizer_prune,run_pruning_epochs,
                    "pruning",after_epoch=lambda epoch:pruning_controller.on_epoch_end())
# Epoch-end masking precedes checkpoints, without an extra schedule refresh.
assert pruning_controller.step==run_pruning_epochs*steps_per_epoch
assert pruning_controller.refresh_count==(2 if SMOKE else 50)


### Finally, we finished pruning. Let's see how mini-batch loss changing over epoch! 

In [ ]:
fig,ax=plt.subplots(figsize=(12,4));ax.semilogy([row["loss"] for row in history_prune])
for epoch_index in range((1 if SMOKE else prune_epoch)-1,run_pruning_epochs,1 if SMOKE else prune_epoch):ax.axvline(epoch_index,color="r",alpha=.2)
ax.set(xlabel="epoch",ylabel="training loss",title="Magnitude pruning: 10% to 90%")
fig.savefig(OUT / "pruned_learning_curve.png");plt.show();plt.close(fig)


### analyze the weights

- let's verify the size of the pruned layer, see if 90% of the weights are zero?

In [ ]:
zero_count=int((hyper_siren_layer.w==0).sum())
weight_count=hyper_siren_layer.w.numel()
sparsity=zero_count/weight_count
assert abs(sparsity-0.90)<=1/weight_count
print({"zero_weights":zero_count,"total_weights":weight_count,"sparsity":sparsity})


- we make prediction on the pruned model

In [ ]:
y_pred=predict(model_for_pruning,input_feature).reshape(200,10)


### after pruning, let's save stripped model to file

In [ ]:
# Strip the pruning controls: the native .pt contains the actual sparse tensor.
pruning_controller.strip()
model_for_export=save_and_check(model_for_pruning,optimizer_prune,"stripped",input_feature[:64])
pruned_file=OUT / "stripped.pt"


In [ ]:
model_for_export=model_for_export.cpu().eval()
example=torch.tensor(input_feature[:32],dtype=torch.float32)
traced=torch.jit.trace(model_for_export,example)
exported_file=OUT / "compressed.torchscript.pt"
torch.jit.save(traced,exported_file)
exported=torch.jit.load(exported_file,map_location="cpu")
with torch.no_grad():
    torch.testing.assert_close(exported(example),model_for_export(example))
    torch.testing.assert_close(exported(torch.tensor(input_feature[:17],dtype=torch.float32)),
                               model_for_export(torch.tensor(input_feature[:17],dtype=torch.float32)))


### Now, we can take a look at how much compression we got

In [ ]:
def get_gzipped_model_size(path):
    import gzip
    return len(gzip.compress(Path(path).read_bytes()))


## Finally, let us try quantazation to maximally compress the size

In [ ]:
# A real native dynamic int8 kernel executes the generated-weight projection.
# The nonlinear SIREN and per-example generated ShapeNet weights stay floating point.
# This is weight/activation dynamic quantization, not a float model saved under an int8 filename.
float_quant_model=copy.deepcopy(model_for_export)
for name,layer in list(float_quant_model.named_modules()):
    if isinstance(layer,HyperLinearForSIREN):
        linear=nn.Linear(layer.w.shape[0],layer.w.shape[1])
        with torch.no_grad():
            linear.weight.copy_(layer.w.T);linear.bias.copy_(layer.b)
        parent_name,_,child_name=name.rpartition(".")
        parent=float_quant_model.get_submodule(parent_name) if parent_name else float_quant_model
        setattr(parent,child_name,linear)
with torch.no_grad():torch.testing.assert_close(float_quant_model(example),model_for_export(example))
quantized_model=torch.ao.quantization.quantize_dynamic(float_quant_model,{nn.Linear},dtype=torch.qint8,inplace=False).eval()
quantized_linears=[layer for layer in quantized_model.modules()
                  if isinstance(layer,torch.ao.nn.quantized.dynamic.Linear)]
assert quantized_linears and all(layer.weight().dtype==torch.qint8 for layer in quantized_linears)
quantized_file=OUT / "quantized_and_pruned.torchscript.pt"
quantized_script=torch.jit.trace(quantized_model,example)
torch.jit.save(quantized_script,quantized_file)
restored_quantized=torch.jit.load(quantized_file,map_location="cpu")
with torch.no_grad():
    torch.testing.assert_close(restored_quantized(example),quantized_model(example),rtol=0,atol=0)
print("Native packed int8 Linear kernels:",len(quantized_linears))


In [ ]:
with torch.no_grad():
    # Full batch predictions are validated after restoring the exported artifact.
    quant_input=torch.tensor(input_feature,dtype=torch.float32)
    y_pred_quant=restored_quantized(quant_input).cpu().numpy().reshape(200,10)
    torch.testing.assert_close(restored_quantized(quant_input),quantized_model(quant_input),rtol=0,atol=0)
assert np.isfinite(y_pred_quant).all()


# Finally, let's compare the compressed result! 

In [ ]:
fig,axs=plt.subplots(3,2,figsize=(12,12))
for column,compressed,label in ((0,y_pred,"pruned 90%"),(1,y_pred_quant,"pruned 90% + int8")):
    for row,values,title in ((0,y_pred_unpruned,"unpruned"),(1,compressed,label),(2,compressed-y_pred_unpruned,"difference")):
        im=axs[row,column].contourf(tt,xx,values,levels=50,cmap="seismic");fig.colorbar(im,ax=axs[row,column]);axs[row,column].set_title(title)
fig.tight_layout();fig.savefig(OUT / "compression_comparison.png");plt.show();plt.close(fig)


In [ ]:
denominator=float(np.sum(y_pred_unpruned**2))
write_metrics({"baseline_epochs":epochs(nepoch),"pruning_epochs":run_pruning_epochs,
    "sparsity":sparsity,"zero_weights":zero_count,"total_weights":weight_count,
    "quantized_linear_count":len(quantized_linears),
    "relative_mse_prune":float(np.sum((y_pred-y_pred_unpruned)**2)/denominator),
    "relative_mse_prune_quant":float(np.sum((y_pred_quant-y_pred_unpruned)**2)/denominator),
    "quantization_mse_vs_pruned":float(np.mean((y_pred_quant-y_pred)**2)),
    "gzip_bytes":{p.name:get_gzipped_model_size(p) for p in (original_file,pruned_file,exported_file,quantized_file)}})
